# Simulation: Generate Combined Noise (Gaussian + Uniform)

This notebook generates simulated clonal data with both Gaussian (Normal) noise and Uniform redistribution noise applied sequentially.

## Process Flow
1. **Load Deterministic Baseline:**
   The deterministic baseline is recovered from the standard dataset by taking the median across noise replicates.
2. **Apply Gaussian Noise first:**
   Standard deviation $\text{dev} = \text{deterministic} \times \frac{P_{\text{Normal}}}{100}$. For zero-count compartments, standard deviation is floored at 1.0 (or 0.01 for Metaclone 9 to prevent random cell regeneration after collapse).
3. **Apply Uniform Redistribution on top:**
   From the Gaussian-noised intermediate data, a percentage $P_{\text{Uniform}}$ of cells are randomly selected and redistributed uniformly across random columns per timepoint block. Total cell count is preserved.

## Noise Levels
Both $P_{\text{Normal}}$ and $P_{\text{Uniform}}$ range from 0% to 100%, every 2% (51 levels each, creating $51 \times 51 = 2601$ combinations per metaclone).

## Output Structure
Save to `Combined_Noise/MC{metaclone_number}/` subdirectory with filenames:
`MC{metaclone_number}_Normal_{dev_noise_pct}_Uniform_{uniform_noise_pct}.csv`


In [3]:
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor

# =========================================================
# Configuration
# =========================================================

STANDARD_DIR = 'simulated_data/Standard'
OUTPUT_DIR = 'simulated_data/Combined_Noise'

NOISE_LEVELS = list(range(0, 101, 2))  # 51 levels

N_CELL_TYPES = 13
N_TIMEPOINTS = 4

# Replicate counts per metaclone (matching the standard dataset)
REPLICATE_COUNTS = {
    1: 15,  2: 30,  3: 50,  4: 40,
    5: 72,  6: 56,  7: 88,  8: 20,
    9: 37, 10: 53, 11: 26, 12: 30
}

print(f"Normal noise levels: {len(NOISE_LEVELS)} levels from {NOISE_LEVELS[0]}% to {NOISE_LEVELS[-1]}%")
print(f"Uniform noise levels: {len(NOISE_LEVELS)} levels from {NOISE_LEVELS[0]}% to {NOISE_LEVELS[-1]}%")
print(f"Total combinations per metaclone: {len(NOISE_LEVELS) * len(NOISE_LEVELS)}")
print(f"Total files to generate across 12 metaclones: {12 * len(NOISE_LEVELS) * len(NOISE_LEVELS)} files")

Normal noise levels: 51 levels from 0% to 100%
Uniform noise levels: 51 levels from 0% to 100%
Total combinations per metaclone: 2601
Total files to generate across 12 metaclones: 31212 files


In [4]:
# =========================================================
# Helper Functions
# =========================================================

def load_deterministic_baseline(meta_id):
    """
    Load standard CSV data and recover the deterministic baseline
    by computing the median across noise replicates.
    """
    csv_path = os.path.join(STANDARD_DIR, f'simulated_clones_meta{meta_id}.csv')
    data = np.loadtxt(csv_path, delimiter=',')
    
    # Recover deterministic baseline via median
    baseline = np.median(data, axis=0)
    baseline = np.round(baseline)
    baseline[baseline < 0] = 0
    return baseline


def apply_gaussian_noise(deterministic, pct, num_replicates, meta_id, rng_seed=42):
    """
    Apply additive Gaussian (normal) noise to the deterministic baseline.
    """
    rng = np.random.RandomState(rng_seed)
    n_features = len(deterministic)
    result = np.zeros((num_replicates, n_features))
    
    noise_fraction = pct / 100.0
    
    for i in range(num_replicates):
        if noise_fraction > 0:
            dev = deterministic * noise_fraction
            # Special case for Metaclone 9: prevent random cell regeneration
            if meta_id == 9:
                dev[dev == 0] = 0.01
            else:
                dev[dev == 0] = 1.0
            noise = rng.randn(n_features) * dev
            noisy = deterministic + noise
        else:
            noisy = deterministic.copy()
        
        noisy = np.round(noisy)
        noisy[noisy < 0] = 0
        result[i] = noisy
    
    return result


def apply_uniform_noise_on_matrix(gaussian_data, pct, rng_seed=42):
    """
    Apply uniform redistribution noise to each row in the gaussian-noised matrix.
    Total cell count per timepoint is preserved.
    """
    rng = np.random.RandomState(rng_seed)
    num_replicates, n_features = gaussian_data.shape
    result = np.zeros_like(gaussian_data)
    
    # Round to integers
    gaussian_int = np.round(gaussian_data).astype(int)
    gaussian_int[gaussian_int < 0] = 0
    
    for i in range(num_replicates):
        row = gaussian_int[i].copy()
        
        for t in range(N_TIMEPOINTS):
            start = t * N_CELL_TYPES
            end = start + N_CELL_TYPES
            block = row[start:end].copy()
            
            total = int(np.sum(block))
            if total == 0:
                continue
            
            n_move = int(np.round(total * pct / 100.0))
            if n_move == 0:
                continue
            
            if n_move >= total:
                # 100% noise: completely random distribution
                new_positions = rng.randint(0, N_CELL_TYPES, size=total)
                row[start:end] = np.bincount(new_positions, minlength=N_CELL_TYPES)
                continue
            
            # Expand block into flat list of cell column indices
            flat_indices = np.repeat(np.arange(N_CELL_TYPES), block)
            
            # Shuffle to randomly select which cells to move
            rng.shuffle(flat_indices)
            
            # Keep first (total - n_move) cells in place
            n_keep = total - n_move
            kept = flat_indices[:n_keep]
            
            # Redistribute the moved cells to uniformly random columns
            new_positions = rng.randint(0, N_CELL_TYPES, size=n_move)
            
            # Reconstruct the row segment
            final = np.concatenate([kept, new_positions])
            row[start:end] = np.bincount(final, minlength=N_CELL_TYPES)
            
        result[i] = row
    return result

print("Functions defined successfully.")


Functions defined successfully.


## Run Combined Noise Simulation Loop
Run the cell below to generate all noise variants for all 12 metaclones.
This writes 31,212 files in `simulated_data/Combined_Noise/MC{mc_idx}/` directory structure using multiple threads.


In [5]:
# =========================================================
# Main Execution Loop
# =========================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

total_files = 0

def save_csv(filepath, data):
    np.savetxt(filepath, data, delimiter=',', fmt='%.4f')

# Use 50 worker threads to write files in parallel to network directory
with ThreadPoolExecutor(max_workers=50) as executor:
    for meta_id in range(1, 13):
        mc_idx = meta_id - 1
        mc_dir_name = f'MC{mc_idx}'
        num_replicates = REPLICATE_COUNTS[meta_id]
        
        print(f'\n{"="*60}')
        print(f'Processing Metaclone {meta_id} -> {mc_dir_name} ({num_replicates} replicates)')
        print(f'{"="*60}')
        
        baseline = load_deterministic_baseline(meta_id)
        mc_out_dir = os.path.join(OUTPUT_DIR, mc_dir_name)
        os.makedirs(mc_out_dir, exist_ok=True)
        
        futures = []
        for g_pct in NOISE_LEVELS:
            # Reusable Gaussian seed dependent on meta_id and g_pct
            g_seed = meta_id * 1000 + g_pct
            gaussian_data = apply_gaussian_noise(baseline, g_pct, num_replicates, meta_id, rng_seed=g_seed)
            
            for u_pct in NOISE_LEVELS:
                # Seed for Uniform noise depends on meta_id, g_pct, and u_pct
                u_seed = meta_id * 100000 + g_pct * 1000 + u_pct + 50000
                combined_data = apply_uniform_noise_on_matrix(gaussian_data, u_pct, rng_seed=u_seed)
                
                filename = f'MC{mc_idx}_Normal_{g_pct}_Uniform_{u_pct}.csv'
                filepath = os.path.join(mc_out_dir, filename)
                
                # Submit write job to executor
                futures.append(executor.submit(save_csv, filepath, combined_data))
                total_files += 1
                
        # Wait for all writes for this metaclone to complete
        for future in futures:
            future.result()
            
        print(f'  Saved {len(NOISE_LEVELS) * len(NOISE_LEVELS)} files for {mc_dir_name}')

print(f'\n{"="*60}')
print(f'All simulations complete!')
print(f'Total files generated: {total_files}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'{"="*60}')



Processing Metaclone 1 -> MC0 (15 replicates)
  Saved 2601 files for MC0

Processing Metaclone 2 -> MC1 (30 replicates)
  Saved 2601 files for MC1

Processing Metaclone 3 -> MC2 (50 replicates)
  Saved 2601 files for MC2

Processing Metaclone 4 -> MC3 (40 replicates)
  Saved 2601 files for MC3

Processing Metaclone 5 -> MC4 (72 replicates)
  Saved 2601 files for MC4

Processing Metaclone 6 -> MC5 (56 replicates)
  Saved 2601 files for MC5

Processing Metaclone 7 -> MC6 (88 replicates)
  Saved 2601 files for MC6

Processing Metaclone 8 -> MC7 (20 replicates)
  Saved 2601 files for MC7

Processing Metaclone 9 -> MC8 (37 replicates)
  Saved 2601 files for MC8

Processing Metaclone 10 -> MC9 (53 replicates)
  Saved 2601 files for MC9

Processing Metaclone 11 -> MC10 (26 replicates)
  Saved 2601 files for MC10

Processing Metaclone 12 -> MC11 (30 replicates)
  Saved 2601 files for MC11

All simulations complete!
Total files generated: 31212
Output directory: simulated_data/Combined_Noise


## Verification and Validation Checks
Check that generated data meets total cell count conservation and shape requirements.


In [6]:
# =========================================================
# Verification: Check a few files and properties
# =========================================================

print("Checking directories and file counts:")
for mc_idx in range(12):
    mc_dir = os.path.join(OUTPUT_DIR, f'MC{mc_idx}')
    if os.path.exists(mc_dir):
        files = os.listdir(mc_dir)
        print(f"  MC{mc_idx}: {len(files)} files")

print()

# Verification of cell count preservation:
# Compare a Gaussian matrix vs Gaussian+Uniform matrix at 50% / 50%
meta_id = 1
mc_idx = meta_id - 1
baseline = load_deterministic_baseline(meta_id)

g_seed = meta_id * 1000 + 50
gaussian_50 = apply_gaussian_noise(baseline, 50, REPLICATE_COUNTS[meta_id], meta_id, rng_seed=g_seed)

filepath_50_50 = os.path.join(OUTPUT_DIR, f'MC{mc_idx}', f'MC{mc_idx}_Normal_50_Uniform_50.csv')
combined_50_50 = np.loadtxt(filepath_50_50, delimiter=',')

# Sum for each timepoint (13 cells block)
for t in range(N_TIMEPOINTS):
    start = t * N_CELL_TYPES
    end = start + N_CELL_TYPES
    g_sum = gaussian_50[:, start:end].sum(axis=1)
    c_sum = combined_50_50[:, start:end].sum(axis=1)
    
    match = np.allclose(g_sum, c_sum)
    print(f"  Timepoint {t} sum preservation match: {match} (Gaussian sum: {g_sum[:3]}, Combined sum: {c_sum[:3]})")

print("\nVerification checks complete!")


Checking directories and file counts:
  MC0: 2601 files
  MC1: 2601 files
  MC2: 2601 files
  MC3: 2601 files
  MC4: 2601 files
  MC5: 2601 files
  MC6: 2601 files
  MC7: 2601 files
  MC8: 2601 files
  MC9: 2601 files
  MC10: 2601 files
  MC11: 2601 files

  Timepoint 0 sum preservation match: True (Gaussian sum: [16. 14.  8.], Combined sum: [16. 14.  8.])
  Timepoint 1 sum preservation match: True (Gaussian sum: [ 97. 200.  99.], Combined sum: [ 97. 200.  99.])
  Timepoint 2 sum preservation match: True (Gaussian sum: [274. 289. 431.], Combined sum: [274. 289. 431.])
  Timepoint 3 sum preservation match: True (Gaussian sum: [356. 399. 492.], Combined sum: [356. 399. 492.])

Verification checks complete!
